# Day 34: GPTQ-Style Round-to-Nearest with Hessian

**Layer:** Implementation | **Prerequisite:** Previous days


## Concept Overview

GPTQ quantizes weights one at a time, using the Hessian diagonal to correct subsequent weights and minimize output error. This converts the per-weight quantization error compensation problem into a series of rank-1 updates.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. GPTQ-Style Round-to-Nearest with Hessian Weighting

GPTQ minimizes the output error after quantizing each weight by solving a second-order optimization. The Hessian diagonal approximates the sensitivity of each weight.


In [ ]:
def gptq_quantize_layer(W, X_calibration, n_bits=4, block_size=128):
    """
    Simplified GPTQ: quantize W minimizing ||WX - W_q X||² using Hessian.
    W: [d_out, d_in]
    X_calibration: [n_samples, d_in]
    """
    d_out, d_in = W.shape
    # Compute Hessian diagonal: H_ii ≈ (1/n) * sum(x_i^2)
    H = (X_calibration**2).mean(dim=0)  # [d_in]
    H = H + 0.01 * H.max()  # damping

    W_q = W.clone().float()
    qmax = 2**(n_bits-1) - 1

    for col in range(0, d_in, block_size):
        col_end = min(col + block_size, d_in)
        W_block = W[:, col:col_end].float()
        H_block = H[col:col_end]

        for j in range(col_end - col):
            w = W_block[:, j]  # [d_out]
            # Quantize
            scale = w.abs().max() / qmax
            w_q = torch.clamp(torch.round(w / scale), -qmax, qmax) * scale
            err = w - w_q  # quantization error [d_out]
            W_q[:, col+j] = w_q
            # Compensate remaining weights in block
            if j + 1 < col_end - col:
                W_block[:, j+1:] -= (err.unsqueeze(1) * H_block[j+1:].unsqueeze(0)) / (H_block[j] + 1e-8)

    return W_q

torch.manual_seed(42)
W = torch.randn(256, 512) * 0.01
X_calib = torch.randn(100, 512)

# Simple INT4 round-to-nearest
qmax = 7  # 4-bit signed
scale_naive = W.abs().max() / qmax
W_naive = torch.clamp(torch.round(W / scale_naive), -qmax, qmax) * scale_naive

# GPTQ-style
W_gptq = gptq_quantize_layer(W, X_calib, n_bits=4)

err_naive = (W - W_naive).abs().mean().item()
err_gptq = (W - W_gptq).abs().mean().item()
print(f'Naive INT4 MAE:  {err_naive:.6f}')
print(f'GPTQ INT4 MAE:   {err_gptq:.6f}')
print(f'GPTQ improvement: {err_naive/err_gptq:.2f}x')


## 2. Output Error Comparison


In [ ]:
# Compare output error on calibration data
out_fp16 = X_calib @ W.T
out_naive = X_calib @ W_naive.T
out_gptq = X_calib @ W_gptq.T

err_naive_out = (out_fp16 - out_naive).abs().mean().item()
err_gptq_out = (out_fp16 - out_gptq).abs().mean().item()
print(f'Naive INT4 output MAE: {err_naive_out:.6f}')
print(f'GPTQ INT4 output MAE:  {err_gptq_out:.6f}')
print(f'Output error reduction: {err_naive_out/err_gptq_out:.2f}x')

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(['Naive INT4', 'GPTQ INT4'], [err_naive_out, err_gptq_out], color=['salmon','steelblue'])
ax.set_ylabel('Mean Absolute Output Error')
ax.set_title('GPTQ vs Naive INT4 Output Quality')
for i, v in enumerate([err_naive_out, err_gptq_out]):
    ax.text(i, v, f'{v:.6f}', ha='center', va='bottom')
plt.savefig('gptq_comparison.png', dpi=100, bbox_inches='tight')
plt.show()


## Experiments

1. Scale GPTQ to a full transformer layer (QKV + FFN). How does total quantization time scale with model size?
2. Try different block sizes (64, 128, 256). How does this affect accuracy vs compute tradeoff?
3. Implement the lazy batch variant: accumulate Hessian over multiple mini-batches.


## Key Takeaways

- See concept overview above for the key points.
- Day 34 complete.
